# 第 7 课作业：拆开 combinational path 与 register edge

这份作业对应 **第 7 课：第一个 RTL 神经元**。

你要把同一个教学 neuron update 拆成两部分：先算 combinational next-state，再在 clock edge 把结果写进 register。

## 先不用代码

先手算：

- `membrane_v=2`、`input_current=1`、`threshold=4`、`reset_value=0`；
- 再把 `membrane_v` 改成 `3`，其他量不变。

分别写出 `candidate`、`spike_next`、`next_v`。第二组为什么会在 candidate 恰好等于 threshold 时 spike？

本题同样只使用不会触发 8-bit overflow 的数值。

## Part A：实现 combinational path

### 这个函数做什么？

`if_neuron_comb()` 对应 RTL 的 `always_comb` 部分。它读取当前 register state 与输入，只计算本次 edge **之前** 可得到的组合结果；它本身不保存状态。

### 输入

- `membrane_v`：当前 register 中的膜电位；
- `input_current`：当前输入；
- `threshold`：spike 阈值；
- `reset_value`：发生 spike 时准备写回的值。

### 输出

返回顺序固定为：

`(candidate, next_v, spike_next)`

其中：

1. `candidate = membrane_v + input_current`；
2. `spike_next` 是布尔值，candidate **达到或超过** threshold 时为 `True`；
3. `next_v` 是下一次 clock edge 准备写入 register 的值；spike 时为 `reset_value`，否则为 `candidate`。

In [ ]:
def if_neuron_comb(
    membrane_v: int,
    input_current: int,
    threshold: int,
    reset_value: int,
) -> tuple[int, int, bool]:
    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: compute candidate, next_v, and spike_next")
    # YOUR CODE ENDS HERE

## Part B：实现 register edge

### 这个函数做什么？

`if_neuron_register_edge()` 对应 `always_ff @(posedge clk)`：在一个上升沿，把已经算好的 `next_v / spike_next` 写进寄存器；同步 reset 在这个 edge 上优先。

### 输入

- `next_v`、`spike_next`：Part A 已经算好的组合结果；
- `rst_n`：同步、低有效 reset；
- `reset_value`：reset 时膜电位写回值。

### 输出

返回顺序固定为：

`(membrane_v_after, spike_after)`

- 当 `rst_n=False`：返回 `(reset_value, False)`；
- 当 `rst_n=True`：返回 `(next_v, spike_next)`。

In [ ]:
def if_neuron_register_edge(
    next_v: int,
    spike_next: bool,
    rst_n: bool,
    reset_value: int,
) -> tuple[int, bool]:
    # YOUR CODE STARTS HERE
    raise NotImplementedError("TODO: apply synchronous reset or store next-state values")
    # YOUR CODE ENDS HERE

## 检查你的实现

grader 会把 combinational path、threshold boundary、register/reset 三类语义分开检查。这样一个概念出错时，不会把另外一部分的反馈一起吞掉。

In [ ]:
# Course infrastructure: make the repository root importable from a notebook subdirectory.
from pathlib import Path
import sys

_repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "exercises" / "grader").is_dir()
    ),
    None,
)
if _repo_root is None:
    raise FileNotFoundError(
        "Could not find the FPGA-FlyBrain repository root. Open this exercise from inside the repository; preferably create a personal work copy with scripts/start_exercise.py first. / 未找到 FPGA-FlyBrain 仓库根目录。请从仓库内部打开此作业 Notebook；推荐先用 scripts/start_exercise.py 创建个人作业副本。"
    )
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from exercises.grader.lesson07 import check

check(
    if_neuron_comb=if_neuron_comb,
    if_neuron_register_edge=if_neuron_register_edge,
    language="zh",
)

## Human Check

1. 为什么 candidate 可以等于 4，而同一个更新结束后的 membrane_v 却是 0？
2. `spike_next` 与 edge 后保存的 `spike` 属于哪个阶段？
3. reset 为什么放在 register edge，而不是修改 combinational candidate？
4. 这份 Python oracle 与真正的 `tutorial_if_neuron.sv` 各自能证明什么、不能证明什么？